# Random train images: fraud vs. bona-fide

Quick eyeball tool. Loads `data/train_labels.csv` (`1 = fraud`, `0 = bona-fide` per CLAUDE.md)
and the local `train/train/{id}.jpeg` images, then shows 10 random images from each class.
No seed is set, so re-running either grid cell below draws a fresh random sample each time.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def resolve_train_dir(data_root: Path) -> Path:
    direct = data_root / "train" / "train"
    if direct.is_dir():
        return direct
    raw = data_root / "raw" / "train" / "train"
    if raw.is_dir():
        return raw
    raise FileNotFoundError(f"couldn't find train images under {data_root} or {data_root / 'raw'}")


TRAIN_DIR = resolve_train_dir(REPO_ROOT / "data")

labels_csv = REPO_ROOT / "data" / "train_labels.csv"
if not labels_csv.exists():
    labels_csv = REPO_ROOT / "data" / "raw" / "train_labels.csv"

labels = pd.read_csv(labels_csv, dtype={"id": str})
labels["path"] = labels["id"].map(lambda i: TRAIN_DIR / f"{i}.jpeg")
labels = labels[labels["path"].map(lambda p: p.exists())].reset_index(drop=True)

fraud_df = labels[labels["label"] == 1].reset_index(drop=True)
bonafide_df = labels[labels["label"] == 0].reset_index(drop=True)

print(f"image dir: {TRAIN_DIR}")
print(f"{len(labels)} labeled train images found locally")
print(f"fraud={len(fraud_df)}  bona-fide={len(bonafide_df)}")

In [ ]:
def show_random_grid(df_subset: pd.DataFrame, title: str, n: int = 10, cols: int = 5) -> None:
    sample = df_subset.sample(n=n).reset_index(drop=True)
    rows = -(-len(sample) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.8))
    axes = axes.flatten()

    for ax, row in zip(axes, sample.itertuples(index=False)):
        try:
            img = Image.open(row.path).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "load\nfailed", ha="center", va="center")
        ax.set_title(f"{row.type}\n{row.id[:8]}", fontsize=7)
        ax.axis("off")

    for ax in axes[len(sample):]:
        ax.axis("off")

    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    plt.show()

## 10 random fraud images (label = 1)

Re-run this cell for a new random sample.

In [ ]:
show_random_grid(fraud_df, "10 random fraud (label=1) train images", n=10)

## 10 random bona-fide images (label = 0)

Re-run this cell for a new random sample.

In [ ]:
show_random_grid(bonafide_df, "10 random bona-fide (label=0) train images", n=10)